# Weighted Main-Experiment Aggregate Forest Plot

This notebook combines the prompt-matched, prompt-mismatched, and out-of-range aggregate summaries into one $3\times4$ forest plot. The rows follow that experiment order, and one shared legend appears above the full figure.

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / 'analyze_results' / 'sd15_recovery_analysis.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the ActiveConditionalGCS project root.')
ANALYZE_ROOT = PROJECT_ROOT / 'analyze_results'
if str(ANALYZE_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYZE_ROOT))

import sd15_recovery_analysis as recovery
recovery = importlib.reload(recovery)

OUTPUT_DIR = PROJECT_ROOT / 'results' / 'weighted' / 'summary' / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCENARIOS = ('prompt_matched', 'prompt_mismatched', 'out_of_range')
SAMPLING_RATIOS = (0.01, 0.02, 0.03, 0.04, 0.05)

In [ ]:
def load_scenario_module(scenario):
    module_path = ANALYZE_ROOT / 'weighted' / scenario / 'analysis.py'
    spec = importlib.util.spec_from_file_location(f'weighted_{scenario}_analysis', module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f'Could not load {module_path}.')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

MODULES = {scenario: load_scenario_module(scenario) for scenario in SCENARIOS}
FRAMES = [MODULES[scenario].load_rows() for scenario in SCENARIOS]
for scenario, frame in zip(SCENARIOS, FRAMES):
    print(f'{scenario}: {len(frame)} rows')

## Combined Aggregate Metrics

From top to bottom, the rows show the in-range prompt-matched, in-range prompt-mismatched, and out-of-range experiments. Metric names appear below the corresponding columns.

In [ ]:
FOREST_OUTPUT = recovery.plot_combined_aggregate_recovery_forest(
    FRAMES,
    expected_sampling_ratios=SAMPLING_RATIOS,
    output_path=OUTPUT_DIR / 'aggregate_metric_forest_combined.pdf',
    show=True,
)
FOREST_OUTPUT